# 🧪 Pipeline de transformation des données météo vers MongoDB Time Series

## 🔍 Objectif

Ce projet unifie plusieurs sources hétérogènes de données météo (stations professionnelles InfoClimat et stations amateurs WeatherUnderground) pour un stockage cohérent dans une base MongoDB Time Series.

---

## 🔁 Étapes de transformation

### 1. Fichiers traités
- `Data_Source1_011024-071024.json` (données InfoClimat, format brut)
- `WeatherUndergroundLaMadeleineFR.xlsx` (données amateurs)
- `WeatherUndergroundIchtegemBE.xlsx` (données amateurs)

### 2. Nettoyage et normalisation
- Conversion des unités :
  - Température : °F → °C
  - Pression : inHg → hPa (`× 33.8639`)
  - Vitesse : mph → km/h
  - Précipitations : in → mm
- Fusion des métadonnées station
- Construction de `dh_utc` (date ISO standard) : ex. `2024-10-01T00:00:00Z`
- Ajout d’un champ `type_station` : `amateur` ou `pro`

### 3. Résultat
- Export vers un fichier JSON plat : `observations_flat.json`
- Ce fichier est directement importable dans une collection MongoDB Time Series avec index sur `dh_utc`.

---

## ⚙️ Fonctionnement du script

- Le script lit les fichiers source (JSON + Excel)
- Utilise des fonctions `safe_*` pour garantir la robustesse du typage
- Ajoute les métadonnées selon la station d’origine
- Vérifie que chaque observation contient au moins une mesure utile
- Produit un JSON nettoyé, sans valeurs nulles ni doublons

---

## 📦 À utiliser ensuite

- Outils d’analyse qualité (ex. cohérence, outliers)
- Import MongoDB via module `ts_crud.py`
- Visualisation avec pandas, matplotlib, etc.



In [ ]:
# 📦 Imports
import pandas as pd
import json
from pathlib import Path
import re

# 📁 Chemins
json_path = Path("data/Data_Source1_011024-071024.json")
xlsx_lm = Path("data/WeatherUndergroundLaMadeleineFR.xlsx")
xlsx_ic = Path("data/WeatherUndergroundIchtegemBE.xlsx")
output_path = Path("data/observations_flat.json")

# -----------------------------
# 🔧 Helpers
# -----------------------------
def safe_str(x):
    s = str(x).replace("\xa0", " ").strip() if x is not None else None
    return s if s else None

def safe_float(x):
    s = safe_str(x)
    if s:
        s = s.replace(",", ".")
        m = re.search(r"-?\d+(\.\d+)?", s)
        return float(m.group(0)) if m else None
    return None

def safe_int(x):
    val = safe_float(x)
    return int(val) if val is not None else None

def parse_date(sheet_name):
    try:
        return pd.to_datetime(sheet_name, dayfirst=True).strftime("%Y-%m-%d")
    except:
        return None

def parse_time_to_hms(val):
    try:
        return pd.to_datetime(val).strftime("%H:%M:%S")
    except:
        return None

def make_dh_utc(date, time):
    return f"{date}T{time}Z" if date and time else None

# -----------------------------
# 📌 Métadonnées stations
# -----------------------------
stations_amateurs = {
    "ILAMAD25": {
        "station_id": "ILAMAD25",
        "station_name": "La Madeleine",
        "ville": "La Madeleine",
        "latitude": 50.659,
        "longitude": 3.07,
        "elevation": 23,
        "hardware": "other",
        "software": "EasyWeatherPro_V5.1.6",
        "type_station": "amateur",
        "source": "Weather Underground (PWS amateur)"
    },
    "IICHTE19": {
        "station_id": "IICHTE19",
        "station_name": "WeerstationBS",
        "ville": "Ichtegem",
        "latitude": 51.092,
        "longitude": 2.999,
        "elevation": 15,
        "hardware": "other",
        "software": "EasyWeatherV1.6.6",
        "type_station": "amateur",
        "source": "Weather Underground (PWS amateur)"
    }
}

# -----------------------------
# 📊 Lecture et nettoyage Excel
# -----------------------------
def clean_excel(path, station_id):
    xls = pd.ExcelFile(path)
    data = []
    for sheet in xls.sheet_names:
        date = parse_date(sheet)
        df = xls.parse(sheet)
        df.columns = [str(c).strip() for c in df.columns]
        for _, row in df.iterrows():
            hms = parse_time_to_hms(row.get("Time"))
            dh_utc = make_dh_utc(date, hms)
            if dh_utc:
                rec = {
                    "dh_utc": dh_utc,
                    "température_°C": safe_float(row.get("Temperature")),
                    "point_de_rosée_°C": safe_float(row.get("Dew Point")),
                    "humidité_%": safe_int(row.get("Humidity")),
                    "pression_hPa": round(safe_float(row.get("Pressure")) * 33.8639, 2)
                        if row.get("Pressure") else None,
                    "vent_direction": safe_str(row.get("Wind")),
                    "vent_moyen_km/h": safe_float(row.get("Speed")),
                    "vent_rafales_km/h": safe_float(row.get("Gust")),
                    "pluie_taux_mm/h": safe_float(row.get("Precip. Rate.")),
                    "pluie_cumulee_mm": safe_float(row.get("Precip. Accum.")),
                    "indice_uv": safe_int(row.get("UV")),
                    "rayonnement_solaire_W/m²": safe_float(row.get("Solar"))
                }
                rec.update(stations_amateurs[station_id])
                data.append(rec)
    return data

# -----------------------------
# 📄 Lecture JSON pro (InfoClimat)
# -----------------------------
with open(json_path, "r", encoding="utf-8") as f:
    json_data = json.load(f)

stations_meta = {
    st["id"]: {
        "station_id": st["id"],
        "station_name": st.get("name"),
        "latitude": st.get("latitude"),
        "longitude": st.get("longitude"),
        "elevation": st.get("elevation"),
        "type_station": "pro",
        "source": st.get("license", {}).get("source")
    } for st in json_data["stations"]
}

json_obs = []
for st_id, entries in json_data.get("hourly", {}).items():
    if st_id == "_params":
        continue
    for entry in entries:
        dh = pd.to_datetime(entry.get("dh_utc"), errors="coerce", utc=True)
        if pd.isna(dh):
            continue
        json_obs.append({
            "dh_utc": dh.strftime("%Y-%m-%dT%H:%M:%SZ"),
            "température_°C": safe_float(entry.get("temperature")),
            "point_de_rosée_°C": safe_float(entry.get("point_de_rosee")),
            "humidité_%": safe_int(entry.get("humidite")),
            "pression_hPa": safe_float(entry.get("pression")),
            "vent_direction": safe_str(entry.get("vent_direction")),
            "vent_moyen_km/h": safe_float(entry.get("vent_moyen")),
            "vent_rafales_km/h": safe_float(entry.get("vent_rafales")),
            "pluie_taux_mm/h": safe_float(entry.get("taux_precipitation")),
            "pluie_cumulee_mm": safe_float(entry.get("pluie_accumulee")),
            "pluie_1h_mm": safe_float(entry.get("pluie_1h")),
            "pluie_3h_mm": safe_float(entry.get("pluie_3h")),
            **stations_meta.get(st_id, {})
        })

# -----------------------------
# 🔀 Fusion et export
# -----------------------------
data_all = json_obs + clean_excel(xlsx_lm, "ILAMAD25") + clean_excel(xlsx_ic, "IICHTE19")

# Retirer champs null
cleaned = []
for d in data_all:
    cleaned.append({k: v for k, v in d.items() if v is not None})

# Export
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(cleaned, f, indent=2, ensure_ascii=False)

print(f"✅ {len(cleaned)} observations exportées vers {output_path}")




# 📘 Pipeline de traitement & importation des observations météo dans MongoDB

## 🗂️ Objectif

Intégrer des données météorologiques hétérogènes (stations professionnelles et amateurs) dans une base de données MongoDB **Time Series**, en assurant la **qualité**, la **cohérence** et la **traçabilité** des transformations.

---

## 🧭 Vue d’ensemble du pipeline

### 1. **Collecte**

* Données initialement disponibles localement (dans un bucket **Airbyte** simulé)
* Fichiers sources :

  * `Data_Source1_011024-071024.json` : données **professionnelles** InfoClimat
  * `WeatherUndergroundLaMadeleineFR.xlsx` : station amateur La Madeleine
  * `WeatherUndergroundIchtegemBE.xlsx` : station amateur Ichtegem

### 2. **Transformation & normalisation**

* **Objectif** : uniformiser les formats, traduire les unités, structurer les observations
* Étapes :

  * Conversion de l'heure (`dh_utc`) au format **ISO 8601**
  * Unification des noms de colonnes en français
  * Conversion d’unités (ex. : pression atmosphérique de inHg → hPa via `×33.8639`)
  * Ajout des **métadonnées stations** (ville, type_station, coordonnées, etc.)
  * Construction d’un **schema pivot unique** pour les deux types de stations

### 3. **Vérification de la qualité des données**

* Script de contrôle (`coherencedata.ipynb`) incluant :

  * Analyse des types, doublons, valeurs manquantes
  * **Détection des valeurs aberrantes** par règle métier (ex: température < -25°C)
  * Résumé par station, jour, type de station
  * Comparaison **stations pro vs amateurs**
  * Option d’interagir pour supprimer ou garder les valeurs aberrantes (par lot)

### 4. **Export JSON**

* Toutes les données vérifiées sont sauvegardées dans :

  * `observations_flat.json`

---

## 🛠️ Import dans MongoDB Time Series

### 5. **Schéma MongoDB**

* Une seule **collection** : `TestObsProEtAmateur`
* Définition en **Time Series** :

  * `timeField`: `"dh_utc"`
  * `metaField`: `"station_id"`
  * Granularité : `"minutes"`
* Exemple de document type (pivot) :

```json
{
  "dh_utc": "2024-10-01T00:04:00Z",
  "station_id": "ILAMAD25",
  "température_°C": 13.4,
  "pression_hPa": 1013.5,
  "type_station": "amateur",
  "ville": "La Madeleine",
  ...
}
```

### 6. **Script interactif d’importation**

* Chargement du fichier JSON
* Vérification des valeurs hors plages
* **Décision interactive** :

  * Supprimer les outliers
  * Conserver toutes les données
  * Garder les outliers pour certaines stations
* Création de la collection MongoDB en Time Series
* Insertion des documents
* Vérification finale de l’import

---

## ✅ Avantages de la solution

* 💡 **Une seule collection unifiée** pour faciliter l’analyse temporelle croisée
* 🔍 **Validation rigoureuse** en amont via règles de qualité métier
* ⚙️ **Scripts refactorisés** pour usage en local ou déploiement automatisé
* 📈 Préparation directe à l’intégration dans des outils d’analyse / visualisation

---

## 📦 Modules requis

Les dépendances Python sont listées dans `requirements.txt` :

```txt
pandas==2.1.0
numpy==1.25.2
openpyxl==3.1.2
pymongo==4.6.0
```

---

## 📤 Export final (POC)

* La base MongoDB locale contient désormais **4950 documents** prêts à être analysés.
* Ces données peuvent être **réinjectées dans Airbyte** pour d'autres tests de synchronisation.




In [ ]:
# 📦 Imports
import json
import pandas as pd
from pathlib import Path
from pymongo import MongoClient
from pprint import pprint

# 📁 Fichier source
DATA_PATH = Path("data/observations_flat.json")

# 🗄️ MongoDB
MONGO_URI = "mongodb://localhost:27017"
DB_NAME = "GreenCoop"
COLLECTION_NAME = "TestObsProEtAmateur"


In [ ]:
# 📥 Chargement JSON
with open(DATA_PATH, "r", encoding="utf-8") as f:
    raw_data = json.load(f)

df = pd.DataFrame(raw_data)

print(f"✅ {len(df)} documents chargés")

# ⏱️ Typage date
df["dh_utc"] = pd.to_datetime(df["dh_utc"], errors="coerce", utc=True)

# 🔢 Colonnes numériques
NUM_COLS = [
    "température_°C", "point_de_rosée_°C", "humidité_%", "pression_hPa",
    "vent_moyen_km/h", "vent_rafales_km/h",
    "pluie_taux_mm/h", "pluie_cumulee_mm",
    "pluie_1h_mm", "pluie_3h_mm",
    "indice_uv", "rayonnement_solaire_W/m²"
]

for col in NUM_COLS:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")


In [ ]:
print("=== Contrôles structurels ===")
print("Dates invalides :", df["dh_utc"].isna().sum())
print("Stations :", df["station_id"].nunique())
print("Types stations :", df["type_station"].value_counts().to_dict())

# 🔁 Doublons
dup = df.duplicated(subset=["station_id", "dh_utc"]).sum()
print("Doublons (station_id + dh_utc) :", dup)


In [ ]:
RANGES = {
    "température_°C": (-25, 45),
    "pression_hPa": (850, 1100),
    "humidité_%": (0, 100),
    "vent_moyen_km/h": (0, 200),
    "vent_rafales_km/h": (0, 250)
}

mask_outlier = pd.Series(False, index=df.index)

for col, (lo, hi) in RANGES.items():
    if col in df.columns:
        mask = df[col].notna() & ((df[col] < lo) | (df[col] > hi))
        mask_outlier |= mask
        print(f"{col} : {mask.sum()} valeurs hors plage")

df_outliers = df[mask_outlier]
df_clean = df[~mask_outlier]

print(f"\n⚠️ Total outliers : {len(df_outliers)}")
print(f"✅ Données valides : {len(df_clean)}")


In [ ]:
if len(df_outliers) > 0:
    print("\nQue faire des valeurs aberrantes ?")
    print("1️⃣ Les SUPPRIMER")
    print("2️⃣ Les CONSERVER")
    print("3️⃣ Conserver uniquement certaines stations")

    choice = input("Votre choix (1/2/3) : ")

    if choice == "1":
        df_final = df_clean
        print("🗑️ Outliers supprimés")

    elif choice == "2":
        df_final = df
        print("⚠️ Outliers conservés")

    elif choice == "3":
        stations = input("Entrer les station_id à CONSERVER (séparées par ,) : ")
        stations = [s.strip() for s in stations.split(",")]
        df_final = pd.concat([
            df_clean,
            df_outliers[df_outliers["station_id"].isin(stations)]
        ])
        print("✂️ Filtrage par station appliqué")

    else:
        raise ValueError("Choix invalide")
else:
    df_final = df


In [ ]:
client = MongoClient(MONGO_URI)
db = client[DB_NAME]

# ⚠️ Suppression si existe (POC)
if COLLECTION_NAME in db.list_collection_names():
    db[COLLECTION_NAME].drop()

# 🕒 Création collection Time Series
db.create_collection(
    COLLECTION_NAME,
    timeseries={
        "timeField": "dh_utc",
        "metaField": "station_id",
        "granularity": "minutes"
    }
)

collection = db[COLLECTION_NAME]
print("✅ Collection Time Series créée")


In [ ]:
# Conversion pour MongoDB
docs = df_final.copy()
docs["dh_utc"] = docs["dh_utc"].dt.to_pydatetime()

records = docs.to_dict(orient="records")

result = collection.insert_many(records)
print(f"🚀 {len(result.inserted_ids)} documents insérés")


In [ ]:
print("\n=== Vérification MongoDB ===")
print("Documents en base :", collection.count_documents({}))

print("Exemple document :")
pprint(collection.find_one({}, {"_id": 0}))


## 📘 Architecture Replica Set – Projet GreenCoop

### 🧱 Objectif
Mettre en place une réplication MongoDB pour la base `GreenCoop` contenant des données météorologiques sous forme de **Time Series**, avec :

- `timeField`: `dh_utc`
- `metaField`: `station_id`

### 🔁 Composition du Replica Set `rsGreenCoop`

| Rôle       | Nom Hôte     | Port   | Droits                         |
|------------|--------------|--------|--------------------------------|
| Primary    | PRIMARY      | 27017  | Lecture + Écriture             |
| Secondary  | ANALYSTE     | 27018  | Lecture seule (data analysts) |
| Clone / Arbiter | SECONDARY_CLONE | 27019  | Peut voter, pas d’écriture (éventuellement promotion si PRIMARY tombe) |

### ❓ Faut-il un arbitre ?
- ✅ Oui si on **n’a que 2 serveurs** physiques, pour éviter un blocage en cas de panne (split brain).
- ❌ Non si on a **3 serveurs complets** → alors `SECONDARY_CLONE` est un vrai nœud répliqué.

### ✅ Avantages du Time Series Replica
- Rejouabilité en cas de panne
- Scalabilité horizontale plus facile
- Données météo critiques : pas de perte

---



In [ ]:
# ================================================
# PowerShell Script - Start MongoDB Replica Set
# ================================================

Write-Host "🚀 Démarrage du Replica Set 'rsGreenCoop'..." -ForegroundColor Cyan

# Chemin vers mongod.exe
$mongod = "C:\Program Files\MongoDB\Server\8.0\bin\mongod.exe"

# Répertoire de base des données
$basePath = "C:\Users\karap\mongodb"

# Nœuds du Replica Set
$nodes = @(
    @{ name = "PRIMARY";       port = 27017; folder = "rs-primary" },
    @{ name = "ANALYSTE";      port = 27018; folder = "rs-analyste" },
    @{ name = "SECONDARY_CLONE"; port = 27019; folder = "rs-clone" }
)

# Création des dossiers si besoin
foreach ($node in $nodes) {
    $dataPath = Join-Path $basePath $node.folder
    if (-Not (Test-Path $dataPath)) {
        Write-Host "📁 Création du dossier $dataPath"
        New-Item -ItemType Directory -Path $dataPath | Out-Null
    }
}

# Démarrage de chaque nœud mongod
foreach ($node in $nodes) {
    $dataPath = Join-Path $basePath $node.folder
    $logPath = Join-Path $dataPath "mongod.log"
    $port = $node.port

    Write-Host "⚙️ Démarrage du nœud ${node.name} sur le port $port..."

    Start-Process -FilePath "$mongod" `
        -ArgumentList "--port", "$port",
                      "--dbpath", "`"$dataPath`"",
                      "--replSet", "rsGreenCoop",
                      "--bind_ip", "localhost",
                      "--logpath", "`"$logPath`"",
                      "--logappend" `
        -WindowStyle Normal
}
